In [ ]:
import importlib.util
import sys
from pathlib import Path

module_path = Path("../inference/data.py").resolve()
spec = importlib.util.spec_from_file_location("data", module_path)
data = importlib.util.module_from_spec(spec)
sys.modules["data"] = data
spec.loader.exec_module(data)

In [ ]:
import h5py
import numpy as np
import matplotlib.pyplot as plt
import os
import flap
import math

In [ ]:
filename = "/home/molnarbalazs/data/BES_ML_modelling/W7X/W7Xinit-Na-E_50.0-t_8-q_2.0-zeff_1.5-tipte_1-mppmi_1-001.hdf5"
with h5py.File(filename) as f:
    d = f["/init_data"][()]
    r_coord = d[:,8:1008]
    density = d[:,1008:2008]
    emission = d[:,2008:3008]

In [ ]:
grid=r_coord[0]
energy=50
species="Na"
ID="wsv1"
zeff=1.5
q=2.0
temperature=np.array(8)
verbose="W7X synthetic data from Miklos Vecsei, 2025-04-29"

In [ ]:
mask=np.all(r_coord==grid,axis=1)

In [ ]:
density.shape

In [ ]:
density=density[mask]
emission=emission[mask]

In [ ]:
tags=[str(i) for i in range(density.shape[0])]

In [ ]:
test_data=data.besInferenceDatapoints(grid=grid,energy=energy,species=species,ID=ID,zeff=zeff,q=q,temperature=temperature,verbose=verbose)

In [ ]:
test_data.add_datapoints_bulk(density, emission, tags)

In [ ]:
plt.plot(test_data.grid,test_data.datapoints[0]["emission"])

In [ ]:
test_data.get_datapoints()

In [ ]:
#test_data.export_to_h5(path_to_dir="/home/molnarbalazs/data/BES_ML_modelling")

In [ ]:
shot='20250521.055'
file_path='/data2/W7-X/processed_data/APDCAM/flap_recon/20250521.055'
file_name_light='20250521.055_4.5000000000045e-06-10.6199945_light_ds_orig_1747865573104.hdf5'
file_name_density='20250521.055_4.5000000000045e-06-10.6199945_dens_1747865573104.hdf5'
file_name_light_recon='20250521.055_4.5000000000045e-06-10.6199945_light_recon_1747865573104.hdf5'

In [ ]:
light=flap.load(os.path.join(file_path,file_name_light))
density=flap.load(os.path.join(file_path,file_name_density))
light_recon=flap.load(os.path.join(file_path,file_name_light_recon))

In [ ]:
time_instances_light_recon=light_recon.coordinate('Time')[0][:,0]
time_instances_density=density.coordinate('Time')[0][:,0]
time_instances_light_recon==time_instances_density

In [ ]:
time_instances_light=light.coordinate('Time')[0][:,0]
mapping = {val: idx for idx, val in enumerate(time_instances_light)}
mask_timeinstance = np.array([mapping[val] for val in time_instances_density])
light_data=light.data[mask_timeinstance,:]

In [ ]:
density_data=density.data
light_recon_data=light_recon.data

In [ ]:
r_coord=light.coordinate('Device R')[0][mask_timeinstance[0]]
mask_samegrid_1=np.all(light.coordinate('Device R')[0]==r_coord,axis=1)[mask_timeinstance]
mask_samegrid_2=np.all(density.coordinate('Device R')[0]==r_coord,axis=1)
light_data=light_data[mask_samegrid_1*mask_samegrid_2]
density_data=density_data[mask_samegrid_1*mask_samegrid_2]
light_recon_data=light_recon_data[mask_samegrid_1*mask_samegrid_2]

In [ ]:
mask_goodrecon=np.sqrt(np.mean((light_data-light_recon_data)**2,axis=1))/np.mean(light_data,axis=1)<0.1
light_data=light_data[mask_goodrecon]
density_data=density_data[mask_goodrecon]
light_recon_data=light_recon_data[mask_goodrecon]
time_instances_density=time_instances_density[mask_goodrecon]

In [ ]:
mask_goodrecon

In [ ]:
grid=r_coord
energy=0
species="Na"
ID="we_"+shot
zeff=0
q=0
temperature=np.array(0)
verbose="W7X experimental data shot 20250521.055, 10Hz averaging, curated for good SPADE recons"
tags=['Time instance ' + str(i) + ' s' for i in time_instances_density]

In [ ]:
grid

In [ ]:
test_data=data.besInferenceDatapoints(grid=grid,energy=energy,species=species,ID=ID,zeff=zeff,q=q,temperature=temperature,verbose=verbose)

In [ ]:
test_data.add_datapoints_bulk(density_data, light_data, tags)

In [ ]:
plt.plot(test_data.grid,test_data.datapoints[10]["emission"])

In [ ]:
test_data.export_to_h5(path_to_dir="/home/molnarbalazs/data/BES_ML_modelling")